# solar_efficiency_prediction

## Import and Load Data

In [117]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

# Load data
train = pd.read_csv('dataset\\train.csv')
test = pd.read_csv('dataset\\test.csv')

In [118]:
train

,id,temperature,irradiance,humidity,panel_age,maintenance_count,soiling_ratio,voltage,current,module_temperature,cloud_coverage,wind_speed,pressure,string_id,error_code,installation_type,efficiency
0,0,7.817315,576.179270,41.24308670850264,32.135501,4.0,0.803199,37.403527,1.963787,13.691147,62.494044,12.82491203459621,1018.8665053152533,A1,NaN,NaN,0.562096
1,1,24.785727,240.003973,1.3596482765960705,19.977460,8.0,0.479456,21.843315,0.241473,27.545096,43.851238,12.012043660984917,1025.6238537572883,D4,E00,dual-axis,0.396447
2,2,46.652695,687.612799,91.26536837560256,1.496401,4.0,0.822398,48.222882,4.191800,43.363708,NaN,1.814399755560454,1010.9226539809573,C3,E00,NaN,0.573776
3,3,53.339567,735.141179,96.19095521176159,18.491582,3.0,0.837529,46.295748,0.960567,57.720436,67.361473,8.736258932034128,1021.8466633134253,A1,NaN,dual-axis,0.629009
4,4,5.575374,12.241203,27.495073003585226,30.722697,6.0,0.551833,0.000000,0.898062,6.786263,3.632000,0.52268384077164,1008.5559577591927,B2,E00,fixed,0.341874
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,19995,16.868428,NaN,93.53031757838667,14.393967,3.0,0.738911,12.147711,3.005355,26.206810,1.733013,12.594122273332914,1018.3744670739436,B2,E02,tracking,0.664907
19996,19996,53.415061,296.970303,93.98571391279083,25.997012,2.0,0.513061,0.000000,0.532119,65.000000,64.558667,0.9769909288128159,1016.081102065643,D4,E00,fixed,0.354070
19997,19997,2.442727,660.328019,37.9689180401391,32.818396,9.0,0.548602,13.047950,4.075498,11.584869,57.730134,4.750937249871706,1009.6844614602336,D4,NaN,tracking,0.419734
19998,19998,NaN,632.760700,43.01470184078199,19.063517,4.0,NaN,0.000000,1.068906,21.149351,78.123689,11.304158443374758,1006.6738746072241,A1,E00,tracking,0.661963


In [119]:
# Basic Obsevation on data set
print(test.isnull().sum().sort_values(ascending=False))
train.isnull().sum().sort_values(ascending=False)

error_code            3611
installation_type     2979
irradiance             615
soiling_ratio          610
maintenance_count      609
panel_age              607
current                587
temperature            582
cloud_coverage         582
module_temperature     580
voltage                547
humidity                 0
id                       0
wind_speed               0
string_id                0
pressure                 0
dtype: int64


error_code            5912
installation_type     5028
maintenance_count     1027
panel_age             1011
soiling_ratio         1010
cloud_coverage        1010
temperature           1001
voltage                993
irradiance             987
module_temperature     978
current                977
id                       0
humidity                 0
pressure                 0
wind_speed               0
string_id                0
efficiency               0
dtype: int64

In [120]:
zero_counts = (train == 0).sum().sort_values(ascending=False)
print("Number of zero values in each column:")
print(zero_counts)

Number of zero values in each column:
voltage               5158
efficiency             631
temperature            389
maintenance_count      349
module_temperature      20
id                       1
humidity                 0
panel_age                0
irradiance               0
current                  0
soiling_ratio            0
wind_speed               0
cloud_coverage           0
pressure                 0
string_id                0
error_code               0
installation_type        0
dtype: int64


In [121]:
zero_power_rows = train[(train['current'] == 0) & (train['voltage'] == 0)]
print(zero_power_rows.shape)
zero_power_rows.head()

(0, 17)


,id,temperature,irradiance,humidity,panel_age,maintenance_count,soiling_ratio,voltage,current,module_temperature,cloud_coverage,wind_speed,pressure,string_id,error_code,installation_type,efficiency


In [122]:
train

,id,temperature,irradiance,humidity,panel_age,maintenance_count,soiling_ratio,voltage,current,module_temperature,cloud_coverage,wind_speed,pressure,string_id,error_code,installation_type,efficiency
0,0,7.817315,576.179270,41.24308670850264,32.135501,4.0,0.803199,37.403527,1.963787,13.691147,62.494044,12.82491203459621,1018.8665053152533,A1,NaN,NaN,0.562096
1,1,24.785727,240.003973,1.3596482765960705,19.977460,8.0,0.479456,21.843315,0.241473,27.545096,43.851238,12.012043660984917,1025.6238537572883,D4,E00,dual-axis,0.396447
2,2,46.652695,687.612799,91.26536837560256,1.496401,4.0,0.822398,48.222882,4.191800,43.363708,NaN,1.814399755560454,1010.9226539809573,C3,E00,NaN,0.573776
3,3,53.339567,735.141179,96.19095521176159,18.491582,3.0,0.837529,46.295748,0.960567,57.720436,67.361473,8.736258932034128,1021.8466633134253,A1,NaN,dual-axis,0.629009
4,4,5.575374,12.241203,27.495073003585226,30.722697,6.0,0.551833,0.000000,0.898062,6.786263,3.632000,0.52268384077164,1008.5559577591927,B2,E00,fixed,0.341874
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,19995,16.868428,NaN,93.53031757838667,14.393967,3.0,0.738911,12.147711,3.005355,26.206810,1.733013,12.594122273332914,1018.3744670739436,B2,E02,tracking,0.664907
19996,19996,53.415061,296.970303,93.98571391279083,25.997012,2.0,0.513061,0.000000,0.532119,65.000000,64.558667,0.9769909288128159,1016.081102065643,D4,E00,fixed,0.354070
19997,19997,2.442727,660.328019,37.9689180401391,32.818396,9.0,0.548602,13.047950,4.075498,11.584869,57.730134,4.750937249871706,1009.6844614602336,D4,NaN,tracking,0.419734
19998,19998,NaN,632.760700,43.01470184078199,19.063517,4.0,NaN,0.000000,1.068906,21.149351,78.123689,11.304158443374758,1006.6738746072241,A1,E00,tracking,0.661963


In [123]:
train.dtypes


id                      int64
temperature           float64
irradiance            float64
humidity               object
panel_age             float64
maintenance_count     float64
soiling_ratio         float64
voltage               float64
current               float64
module_temperature    float64
cloud_coverage        float64
wind_speed             object
pressure               object
string_id              object
error_code             object
installation_type      object
efficiency            float64
dtype: object

In [124]:
train.describe()

,id,temperature,irradiance,panel_age,maintenance_count,soiling_ratio,voltage,current,module_temperature,cloud_coverage,efficiency
count,20000.000000,18999.000000,19013.000000,18989.000000,18973.000000,18990.000000,19007.000000,19023.000000,19022.000000,18990.000000,20000.000000
mean,9999.500000,25.077241,501.273896,17.509758,4.012070,0.698879,16.242251,1.713396,29.923807,51.378575,0.510260
std,5773.647028,12.513129,250.926590,10.097557,2.002268,0.172244,17.889031,1.152953,12.125405,48.473664,0.140420
min,0.000000,0.000000,-597.278646,0.001264,0.000000,0.400149,0.000000,0.000054,0.000000,0.000244,0.000000
25%,4999.750000,16.853522,332.227277,8.777905,3.000000,0.550654,0.000000,0.772311,21.522124,25.081618,0.445613
50%,9999.500000,24.720345,499.654730,17.497731,4.000000,0.697663,12.350138,1.558413,29.857669,49.704133,0.515709
75%,14999.250000,32.848917,668.416734,26.340761,5.000000,0.847838,26.557322,2.474744,38.094943,75.052824,0.590324
max,19999.000000,147.394168,1537.810349,34.998379,15.000000,0.999949,494.279016,7.315597,65.000000,1000.000000,0.987066


## Need to handle Following things:
1. null value
2. 0 value
3. object into float and round off

1. lets convert the string into float and round-of till 5 decimal.

In [125]:
# Check all columns for non-numeric values
# for col in train.columns:
#     if train[col].dtype == 'object':
#         print(f"Non-numeric values in '{col}':")
        # print(train[col].unique())
        
        
num_cols = train.select_dtypes(include=['float64', 'int64']).columns
cat_cols = train.select_dtypes(include='object').columns

print("Categorical Value\n",list(cat_cols))
print("Numeric Value\n",list(num_cols))


Categorical Value
 ['humidity', 'wind_speed', 'pressure', 'string_id', 'error_code', 'installation_type']
Numeric Value
 ['id', 'temperature', 'irradiance', 'panel_age', 'maintenance_count', 'soiling_ratio', 'voltage', 'current', 'module_temperature', 'cloud_coverage', 'efficiency']


In [126]:
# lets handle these value ['humidity', 'wind_speed', 'pressure']
# Now apply convert test and train into float and  check na value.
for col in ['humidity', 'wind_speed', 'pressure']: 
    # Step 1: Convert 'humidity' column to float
    train[col] = pd.to_numeric(train[col], errors='coerce')
    test[col] = pd.to_numeric(train[col], errors='coerce')

    # Step 2: Round the values to 5 decimal places
    train[col] = train[col].round(5)
    test[col] = test[col].round(5)


In [127]:
# Basic Obsevation on data set
print(test.isnull().sum().sort_values(ascending=False))
train.isnull().sum().sort_values(ascending=False)

error_code            3611
installation_type     2979
irradiance             615
soiling_ratio          610
maintenance_count      609
panel_age              607
current                587
temperature            582
cloud_coverage         582
module_temperature     580
voltage                547
humidity                83
pressure                79
wind_speed              69
id                       0
string_id                0
dtype: int64


error_code            5912
installation_type     5028
maintenance_count     1027
panel_age             1011
soiling_ratio         1010
cloud_coverage        1010
temperature           1001
voltage                993
irradiance             987
module_temperature     978
current                977
pressure               135
humidity               127
wind_speed             119
id                       0
string_id                0
efficiency               0
dtype: int64

## Data Cleaning

In [128]:
# Data Cleaning
train.fillna({
    'temperature': train['temperature'].mean(),
    'irradiance': train['irradiance'].mean(),
    'panel_age': train['panel_age'].median(),
    'maintenance_count': train['maintenance_count'].median(),
    'soiling_ratio': train['soiling_ratio'].median(),
    'voltage': train['voltage'].mean(),
    'current': train['current'].mean(),
    'module_temperature': train['module_temperature'].mean(),
    'cloud_coverage': train['cloud_coverage'].median(),
    'error_code': train['error_code'].mode()[0],
    'installation_type': train['installation_type'].mode()[0],
    
}, inplace=True)

test.fillna({
    'temperature': test['temperature'].mean(),
    'irradiance': test['irradiance'].mean(),
    'panel_age': test['panel_age'].median(),
    'maintenance_count': test['maintenance_count'].median(),
    'soiling_ratio': test['soiling_ratio'].median(),
    'voltage': test['voltage'].mean(),
    'current': test['current'].mean(),
    'module_temperature': test['module_temperature'].mean(),
    'cloud_coverage': test['cloud_coverage'].median(),
    'error_code': test['error_code'].mode()[0],
    'installation_type': test['installation_type'].mode()[0]
}, inplace=True)

## Feature Engineering

In [129]:
# Feature Engineering
for df in [train, test]:
    df['power_output'] = df['voltage'] * df['current']
    df['temp_diff'] = df['module_temperature'] - df['temperature']
    df['irradiance_per_cloud'] = df['irradiance'] / (df['cloud_coverage'] + 1)
    
train['power_output'] = train['voltage'] * train['current']
train['temp_diff'] = train['module_temperature'] - train['temperature']
train['irradiance_per_cloud'] = train['irradiance'] / (train['cloud_coverage'] + 1)

    
test['power_output'] = test['voltage'] * test['current']
test['temp_diff'] = test['module_temperature'] - test['temperature']
test['irradiance_per_cloud'] = test['irradiance'] / (test['cloud_coverage'] + 1)

In [130]:
# Check all columns for non-numeric values
for col in train.columns:
    if train[col].dtype == 'object':
        print(f"Non-numeric values in '{col}':")
        print(train[col].unique())
        
        
num_cols = train.select_dtypes(include=['float64', 'int64']).columns
cat_cols = train.select_dtypes(include='object').columns

print(cat_cols)
print(num_cols)


Non-numeric values in 'string_id':
['A1' 'D4' 'C3' 'B2']
Non-numeric values in 'error_code':
['E00' 'E01' 'E02']
Non-numeric values in 'installation_type':
['tracking' 'dual-axis' 'fixed']
Index(['string_id', 'error_code', 'installation_type'], dtype='object')
Index(['id', 'temperature', 'irradiance', 'humidity', 'panel_age',
       'maintenance_count', 'soiling_ratio', 'voltage', 'current',
       'module_temperature', 'cloud_coverage', 'wind_speed', 'pressure',
       'efficiency', 'power_output', 'temp_diff', 'irradiance_per_cloud'],
      dtype='object')


In [131]:
# Now apply convert test and train into float and  check na value.
for col in ['humidity', 'wind_speed', 'pressure']: 
    # Step 1: Convert 'humidity' column to float
    train[col] = pd.to_numeric(train[col], errors='coerce')

    # Step 2: Round the values to 4 decimal places
    train[col] = train[col].round(5)
    
    
# Now apply convert test and train into float and  check na value.
for col in ['humidity', 'wind_speed', 'pressure']: 
    # Step 1: Convert 'humidity' column to float
    test[col] = pd.to_numeric(test[col], errors='coerce')

    # Step 2: Round the values to 4 decimal places
    test[col] = test[col].round(5)
    
    

In [132]:
for col in ['humidity', 'wind_speed', 'pressure']:
    median_value = train[col].median()
    train[col].fillna(median_value, inplace=True)
    test[col].fillna(median_value, inplace=True)

C:\Users\vidha\AppData\Local\Temp\ipykernel_48764\2228389947.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train[col].fillna(median_value, inplace=True)
C:\Users\vidha\AppData\Local\Temp\ipykernel_48764\2228389947.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exampl

In [148]:
# Drop rows with 0 or missing efficiency
print(train.shape)
train = train[train['efficiency'] > 0].copy()
train.shape

(20000, 20)


(19369, 20)

In [149]:
# Encode categorical features
cat_cols = ['string_id', 'error_code', 'installation_type']
for col in cat_cols:
    train[col] = train[col].astype('category').cat.codes
    test[col] = test[col].astype('category').cat.codes

# Feature selection
drop_cols = ['id', 'efficiency']
X = train.drop(columns=drop_cols)
y = train['efficiency']
X_test = test.drop(columns=['id'])

# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


In [83]:
X_train

,temperature,irradiance,humidity,panel_age,maintenance_count,soiling_ratio,current,module_temperature,cloud_coverage,wind_speed,pressure,string_id,error_code,installation_type,power_output,temp_diff,irradiance_per_cloud
5894,26.795761,506.418613,3.07276,33.133493,2.0,0.497343,1.062653,33.530870,56.312859,14.88279,1008.44503,2,0,2,14.991693,6.735110,8.836038
3728,25.077241,570.521973,36.19069,9.114098,1.0,0.507641,0.344663,39.732024,81.174126,3.32688,1022.41151,3,0,1,4.724096,14.654783,6.942842
8958,32.700179,454.975175,73.93501,1.401363,4.0,0.877871,0.813439,40.173099,46.808396,13.13401,1009.37686,3,1,2,9.803852,7.472920,9.516637
7671,43.300607,808.460615,33.36382,8.768143,2.0,0.504152,3.252233,50.431555,49.704133,11.02750,1010.41039,2,0,2,17.335443,7.130948,15.944669
5999,19.841999,154.309598,61.30780,17.497731,1.0,0.839368,1.880971,20.564514,84.813474,4.46317,1006.77390,3,0,2,25.781416,0.722515,1.798198
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11284,37.753249,456.247659,3.12009,17.497731,7.0,0.690635,2.095863,45.414556,59.049390,4.46680,1002.52669,2,0,0,72.282249,7.661307,7.597873
11964,56.718243,302.161983,83.33497,33.152380,4.0,0.697663,2.078109,55.380695,90.749564,3.34718,1021.59811,0,1,1,25.361940,-1.337548,3.293334
5390,36.906454,901.102499,97.98664,17.372952,4.0,0.727222,2.424920,44.093709,84.495256,11.41003,1027.57736,1,0,2,61.256250,7.187255,10.539795
860,37.747638,254.495145,74.06064,7.353131,4.0,0.806019,0.323723,45.788727,83.177460,3.52266,1001.36190,3,0,1,3.594983,8.041089,3.023317


In [154]:
model = CatBoostRegressor(iterations=150, learning_rate=0.1, depth=5, random_seed=42, verbose=False)
model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)

In [155]:

# Validation score
y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
score = (1 - rmse) * 100
print(f"RMSE: {rmse:.4f}, Score: {score:.5f}")


RMSE: 0.0452, Score: 95.47738


In [153]:
model = CatBoostRegressor(iterations=300, learning_rate=0.07, depth=5, random_seed=42, verbose=False)
model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)

# Validation score
y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
score = (1 - rmse) * 100
print(f"RMSE: {rmse:.4f}, Score: {score:.8f}")
# score before volatage is zero: RMSE: 0.1060, Score: 89.39525848

RMSE: 0.0450, Score: 95.49825499


In [152]:
model = CatBoostRegressor(iterations=300, learning_rate=0.07, depth=5, random_seed=42, verbose=False)
model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)

# Validation score
y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
score = (1 - rmse) * 100
print(f"RMSE: {rmse:.4f}, Score: {score:.8f}")

RMSE: 0.0450, Score: 95.49825499


In [88]:
# Case 3
model = CatBoostRegressor(iterations=300, learning_rate=0.07, depth=5, random_seed=42, verbose=False)
model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)

# Validation score
y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
score = (1 - rmse) * 100
print(f"RMSE: {rmse:.4f}, Score: {score:.8f}")

RMSE: 0.1060, Score: 89.39959316


In [146]:
# Case 4 after droping less coreleted values
model = CatBoostRegressor(iterations=300, learning_rate=0.07, depth=5, random_seed=42, verbose=False)
model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)

# Validation score
y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
score = (1 - rmse) * 100
print(f"RMSE: {rmse:.4f}, Score: {score:.8f}")

RMSE: 0.1061, Score: 89.38937456


In [53]:
model = CatBoostRegressor(iterations=300, learning_rate=0.06, depth=5, random_seed=42, verbose=False)
model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)

# Validation score
y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
score = (1 - rmse) * 100
print(f"RMSE: {rmse:.4f}, Score: {score:.8f}")

RMSE: 0.1061, Score: 89.39311855


In [139]:
from sklearn.model_selection import GridSearchCV

params = {
    'depth': [4, 6, 8],
    'learning_rate': [0.03, 0.05, 0.1],
    'iterations': [200, 300, 500]
}

cb = CatBoostRegressor(random_seed=42, verbose=False)

grid = GridSearchCV(cb, params, cv=3, scoring='neg_mean_squared_error')
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)

# 1. Best params: {'depth': 4, 'iterations': 500, 'learning_rate': 0.03}

KeyboardInterrupt: 

In [156]:


# 1. Best params: {'depth': 4, 'iterations': 500, 'learning_rate': 0.03}
model = CatBoostRegressor(iterations=500, learning_rate=0.03, depth=4, random_seed=42, verbose=False)
model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)

# Validation score
y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
score = (1 - rmse) * 100
print(f"RMSE: {rmse:.4f}, Score: {score:.8f}")

RMSE: 0.0456, Score: 95.44487093


In [157]:

from datetime import datetime
# Get current date and time
now = datetime.now()
timestamp = now.strftime("%H%M%S")
print(f"submission_{timestamp}")

# Final prediction
preds = model.predict(X_test)
submission = pd.DataFrame({'id': test['id'], 'efficiency': preds})

submission.to_csv(f'submission_{timestamp}.csv', index=False)

submission_232046


In [86]:
submission.shape

(12000, 2)

In [63]:
train.to_csv(f'cleaned_train{timestamp}.csv', index=False)

In [79]:
print("Zero voltage count:", (test['voltage'] == 0).sum())
print("Total rows:",len(test))

Zero voltage count: 0
Total rows: 12000


In [66]:
print("Zero current count:", (train['current'] == 0).sum())
print("Total rows:",len(train))

Zero current count: 0
Total rows: 20000


In [68]:
print("Zero current count:", (train['power_output'] == 0).sum())
print("Total rows:",len(train))

Zero current count: 5158
Total rows: 20000


In [71]:
voltage_median = train['voltage'].median()
train['voltage'] = train['voltage'].replace(0,voltage_median)
test['voltage'] = test['voltage'].replace(0,voltage_median)


In [90]:
zero_counts = (train == 0).sum().sort_values(ascending=False)
print("Number of zero values in each column:")
print(zero_counts)

Number of zero values in each column:
error_code              11889
installation_type        4915
string_id                4902
efficiency                631
temperature               389
maintenance_count         349
module_temperature         20
temp_diff                  17
id                          1
irradiance                  0
humidity                    0
soiling_ratio               0
wind_speed                  0
cloud_coverage              0
current                     0
voltage                     0
panel_age                   0
pressure                    0
power_output                0
irradiance_per_cloud        0
dtype: int64
